# TRL vs ms-swift 框架对比

本文档从以下几个角度对比两个框架：
1. **语法风格** — CLI vs Python API
2. **启动方法与配置技巧** — 命令行参数 vs 代码配置
3. **Model 加载方式** — 模型加载与 LoRA 配置
4. **Dataset 加载方式** — 数据格式与预处理
5. **针对 mol-llm 项目的建议**

## 1. 语法风格对比

### TRL: 基于 HuggingFace 生态的 Python API

TRL 是 HuggingFace 官方的训练框架，深度集成 `transformers` + `peft` + `datasets`。

**核心特点**:
- 纯 Python API，通过 `TrlParser` 解析 YAML/JSON 配置
- 使用 `dataclass` 定义配置（`ScriptArguments`, `SFTConfig`, `ModelConfig`）
- 直接操作 `Dataset` 对象，灵活但需要自己写数据加载逻辑
- 与 HuggingFace 生态无缝衔接

### ms-swift: 一体化 CLI + Python API

ms-swift 是 ModelScope 社区的全栈框架，覆盖训练、推理、评估、量化、部署。

**核心特点**:
- **CLI 优先**（推荐）：`swift sft` / `swift rlhf` / `swift infer` 一条命令搞定
- 内置 150+ 数据集、600+ 模型模板，开箱即用
- 自动处理 template、tokenize、packing、multimodal
- 也支持 Python API（`sft_main(SftArguments(...))`）

## 2. 启动方法与配置技巧

### TRL 启动方式

**方式一：命令行 + YAML 配置**
```bash
# 当前项目的 sft.py 使用 TrlParser 解析配置
python scripts/sft.py \
    --config_file configs/model_training_schema.yaml \
    --model_name_or_path Qwen/Qwen3-4B \
    --dataset_name kdeng03/mol-rep-conversion-v0 \
    --output_dir ckpt/qwen3_4b_sft \
    --use_peft true \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --learning_rate 1e-4 \
    --num_train_epochs 2 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4
```

**方式二：纯 Python**
```python
from trl import TrlParser, ScriptArguments, SFTConfig, ModelConfig, SFTTrainer, get_peft_config

parser = TrlParser([ScriptArguments, SFTConfig, ModelConfig])
script_args, sft_config, model_config = parser.parse_args_and_config()

peft_config = get_peft_config(model_config)
trainer = SFTTrainer(
    model=model_config.model_name_or_path,
    args=sft_config,
    train_dataset=training_dataset,
    peft_config=peft_config,
)
trainer.train()
```

**配置技巧**:
- TRL 的配置分散在三个 dataclass 中，需要自己组装
- `max_length = None` 是 VLM 训练的特殊处理
- 数据集加载需要自己写 `_load_training_dataset()` 函数

---

### ms-swift 启动方式

**方式一：CLI（推荐）**
```bash
CUDA_VISIBLE_DEVICES=0 \
swift sft \
    --model Qwen/Qwen3-4B-Instruct-2507 \
    --tuner_type lora \
    --dataset 'kdeng03/mol-rep-conversion-v0' \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --per_device_train_batch_size 1 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --gradient_accumulation_steps 16 \
    --eval_steps 50 \
    --save_steps 50 \
    --save_total_limit 2 \
    --logging_steps 5 \
    --max_length 2048 \
    --output_dir output \
    --warmup_ratio 0.05 \
    --dataloader_num_workers 4
```

**方式二：Python API**
```python
from swift import sft_main, SftArguments

result = sft_main(SftArguments(
    model='Qwen/Qwen3-4B-Instruct-2507',
    tuner_type='lora',
    dataset=['kdeng03/mol-rep-conversion-v0'],
    torch_dtype='bfloat16',
    num_train_epochs=1,
    learning_rate=1e-4,
    lora_rank=8,
    lora_alpha=32,
    target_modules='all-linear',
    output_dir='output',
))
```

**方式三：Web-UI**
```bash
SWIFT_UI_LANG=en swift web-ui
```

**配置技巧**:
- `--use_hf true` 切换到 HuggingFace 下载（默认 ModelScope）
- `--dataset 'path#num'` 语法快速采样，如 `'dataset#500'`
- `--target_modules all-linear` 自动选择所有 linear 层
- 内置 template 自动处理 chat format，无需手动拼接
- `--packing true` 提升 GPU 利用率
- `--deepspeed zero2` 一键启用 DeepSpeed

## 3. Model 加载方式对比

### TRL: 手动加载模型

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# 1. 加载模型
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

# 2. 配置 LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                    "up_proj", "down_proj", "gate_proj"],
    task_type="CAUSAL_LM",
)

# 3. 应用 LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. VLM 需要额外处理
# from transformers import AutoProcessor, AutoModelForImageTextToText
# model = AutoModelForImageTextToText.from_pretrained(...)
# processor = AutoProcessor.from_pretrained(...)
```

**特点**:
- 需要手动处理模型加载、tokenizer、processor
- VLM 需要额外加载 `AutoModelForImageTextToText`
- LoRA 配置完全手动
- 灵活但繁琐

---

### ms-swift: 一行命令搞定

```bash
# CLI 自动处理模型加载 + template + LoRA
swift sft \
    --model Qwen/Qwen3-4B-Instruct-2507 \
    --tuner_type lora \
    --lora_rank 16 \
    --lora_alpha 32 \
    --target_modules all-linear
```

**Python API 等价写法**:
```python
from swift import get_model_processor, get_template
from peft import LoraConfig, get_peft_model

# 1. 一行加载模型 + tokenizer + processor
model, tokenizer = get_model_processor("Qwen/Qwen3-4B-Instruct-2507")

# 2. 一行获取 template（自动处理 chat format）
template = get_template(tokenizer)

# 3. 配置 LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",  # 也支持列表
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
```

**特点**:
- `get_model_processor()` 自动识别模型类型（LLM / VLM / MoE）
- `get_template()` 自动匹配 chat template
- `--tuner_type lora` 一行启用 LoRA
- `--target_modules all-linear` 自动选择所有 linear 层
- 支持 `full`, `lora`, `qlora`, `adapter` 等多种 tuner

---

### LoRA 配置参数对比

| 参数 | TRL | ms-swift | 说明 |
|------|-----|----------|------|
| Rank | `lora_r` | `lora_rank` | 推荐 8/16/32 |
| Alpha | `lora_alpha` | `lora_alpha` | 推荐 2x rank |
| Dropout | `lora_dropout` | `lora_dropout` | 小数据建议 0.05~0.1 |
| Target Modules | `lora_target_modules` | `target_modules` | `all-linear` 或列表 |
| Tuner Type | `use_peft` (bool) | `tuner_type` (str) | lora/qlora/full/adapter |

**针对 4B 模型 + 3.5k 样本的建议**:
- `r=16, alpha=32, dropout=0.1` 是比较稳妥的 baseline
- `target_modules=all-linear` 确保有足够容量学习复杂映射
- 如果显存紧张，先用 `["q_proj", "v_proj"]` 做快速验证

## 4. Dataset 加载方式对比

### TRL: 手动加载和处理数据集

```python
from datasets import load_dataset, Dataset

def _load_training_dataset(script_args, sft_config, model_config):
    """需要自己写数据加载函数"""
    # 1. 从 HF 加载
    dataset = load_dataset(script_args.dataset_name, split="train")
    
    # 2. 需要自己处理格式转换
    # TRL 期望的格式: {"prompt": ..., "completion": ...} 或 messages 格式
    def convert_to_messages(example):
        return {
            "messages": [
                {"role": "user", "content": example["prompt"]},
                {"role": "assistant", "content": example["completion"]},
            ]
        }
    
    dataset = dataset.map(convert_to_messages)
    
    # 3. VLM 需要额外处理 image 字段
    # 如果数据集中有图片，需要确保格式正确
    
    return dataset
```

**TRL 支持的数据格式**:
```python
# 格式 1: prompt/completion
{"prompt": "Convert SMILES to InChI:", "completion": "InChI=1S/..."}

# 格式 2: messages (推荐)
{"messages": [
    {"role": "user", "content": "Convert SMILES to InChI:"},
    {"role": "assistant", "content": "InChI=1S/..."}
]}

# 格式 3: text (纯文本)
{"text": "User: Convert SMILES to InChI:\nAssistant: InChI=1S/..."}
```

**问题**:
- 需要自己写 `map` 函数转换格式
- VLM 的 image 字段需要特殊处理
- 多任务混合需要自己 `concatenate_datasets`
- 按分子 split 需要自己实现

---

### ms-swift: 自动处理数据格式

**方式一：直接指定路径（推荐）**
```bash
# 支持 json, jsonl, csv, txt, parquet
swift sft \
    --dataset /path/to/mol-rep-conversion-v0.jsonl \
    --columns 'messages'  # 可选：列名映射
```

**方式二：HF/ModelScope Dataset ID**
```bash
swift sft \
    --dataset kdeng03/mol-rep-conversion-v0 \
    --use_hf true  # 从 HF 下载（默认 ModelScope）
```

**方式三：快速采样**
```bash
# 只取前 500 条
swift sft \
    --dataset 'kdeng03/mol-rep-conversion-v0#500'
```

**方式四：多数据集混合**
```bash
# 自动 concatenate，支持不同采样比例
swift sft \
    --dataset 'dataset1#500' 'dataset2#300' 'dataset3#200'
```

**ms-swift 标准数据格式**:
```json
// SFT 标准格式（messages）
{"messages": [
    {"role": "user", "content": "Convert SMILES to InChI:"},
    {"role": "assistant", "content": "InChI=1S/..."}
]}

// 多模态格式（+ images/videos/audios）
{"messages": [
    {"role": "user", "content": "<image>What is the SMILES of this molecule?"},
    {"role": "assistant", "content": "CCO"}
], "images": ["/path/to/mol_image.png"]}

// DPO/RLHF 格式
{"messages": [...], "rejected_response": "wrong answer"}
```

**AutoPreprocessor 自动识别的格式**:
| 输入格式 | 字段名 | 自动转换 |
|---------|--------|---------|
| Messages | `messages` | ✅ 标准格式 |
| ShareGPT | `conversation` | ✅ 自动转 messages |
| Alpaca | `instruction`, `input`, `output` | ✅ 自动转 messages |
| Query-Response | `query`, `response` | ✅ 自动转 messages |

**特点**:
- 自动识别数据格式并转换
- 内置 150+ 数据集，一行命令使用
- 多数据集混合只需空格分隔
- 自动处理 multimodal 字段（images, videos, audios）
- 支持 `dataset_info.json` 注册自定义数据集

---

### 针对 mol-llm 项目的数据加载对比

**TRL 方式（当前项目）**:
```python
# 需要自己写：
# 1. 加载 HF dataset
# 2. 按分子 split（80 train / 20 test）
# 3. 转换格式为 messages
# 4. 混合多个任务子集
# 5. VLM 需要处理 image 字段

def _load_training_dataset(script_args, sft_config, model_config):
    # 伪代码
    dataset = load_dataset("kdeng03/mol-rep-conversion-v0", split="train")
    
    # 按分子 split
    mol_ids = dataset["mol_id"].unique()
    train_mols = mol_ids[:80]
    test_mols = mol_ids[80:]
    
    train_dataset = dataset.filter(lambda x: x["mol_id"] in train_mols)
    test_dataset = dataset.filter(lambda x: x["mol_id"] in test_mols)
    
    # 转换格式
    def to_messages(example):
        return {"messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["completion"]}
        ]}
    
    train_dataset = train_dataset.map(to_messages)
    return train_dataset
```

**ms-swift 方式**:
```bash
# 1. 准备数据（按任务分文件）
# data/mol-conversion-qa/translation.jsonl
# data/mol-conversion-qa/understanding.jsonl
# data/mol-conversion-qa/normalization.jsonl

# 2. 一条命令混合训练
swift sft \
    --model Qwen/Qwen3-4B \
    --tuner_type lora \
    --dataset \
        'data/mol-conversion-qa/translation.jsonl' \
        'data/mol-conversion-qa/understanding.jsonl' \
        'data/mol-conversion-qa/normalization.jsonl' \
    --output_dir output

# 3. 或者混合所有任务
swift sft \
    --dataset 'data/mol-conversion-qa/all.jsonl'
```

## 5. 针对 mol-llm 项目的建议

### 当前项目状态

**已有**:
- `scripts/sft.py` — 基于 TRL 的 SFT 训练脚本
- `scripts/inference.py` — 推理脚本
- `scripts/gen_mol_reps.py` — Stage 1: 原始表示生成
- `scripts/gen_mol_rep_conversion.py` — Stage 2: 训练任务生成
- `configs/mol_rep_schema.yaml` — 表示 schema
- `configs/mol_rep_conversion_schema.yaml` — 任务 schema
- `configs/model_training_schema.yaml` — 训练配置（待完善）

**数据**:
- `kdeng03/mol-reps-v0` — Stage 1 原始数据
- `kdeng03/mol-rep-ocr-v0` — OCR 任务数据（含 image）
- `kdeng03/mol-rep-conversion-v0` — Conversion 任务数据（纯文本）

---

### 框架选择建议

| 维度 | TRL | ms-swift | 推荐 |
|------|-----|----------|------|
| **学习曲线** | 中等（需熟悉 HF 生态） | 低（CLI 一行命令） | ms-swift |
| **灵活性** | 高（完全 Python 控制） | 中（CLI 受限，但 Python API 也支持） | TRL |
| **VLM 支持** | 需要手动处理 processor | 自动处理 template + multimodal | ms-swift |
| **多数据集混合** | 需要自己 `concatenate_datasets` | 命令行空格分隔 | ms-swift |
| **GRPO/RL** | 需要自己写 reward function | 内置 GRPO/DAPO/GSPO 等 | ms-swift |
| **推理加速** | 需要自己集成 vLLM | 内置 vLLM/SGLang/LMDeploy | ms-swift |
| **生态兼容** | HuggingFace 原生 | 兼容 HF，但默认 ModelScope | TRL |
| **社区活跃度** | HF 官方维护 | ModelScope 社区，中文文档好 | 平手 |

---

### 具体建议

#### 方案 A：继续使用 TRL（当前方案）

**优点**:
- 已有代码基础，改动小
- 完全控制数据加载和训练流程
- 与 HuggingFace 生态无缝衔接

**需要补充的工作**:
1. 完善 `_load_training_dataset()` 函数
2. 实现按分子 split 逻辑
3. 实现多任务混合逻辑
4. 处理 VLM 的 image 字段
5. 完善 `model_training_schema.yaml`

**适合场景**: 需要高度定制化的实验设计

---

#### 方案 B：切换到 ms-swift

**优点**:
- 一条命令启动训练
- 自动处理 template、tokenize、multimodal
- 内置 GRPO 等 RL 算法
- 内置推理加速（vLLM/SGLang）
- 中文文档完善

**需要的工作**:
1. 将数据转换为 ms-swift 标准格式（messages）
2. 编写 shell 脚本替代 `sft.py`
3. 学习 ms-swift 的 CLI 参数

**适合场景**: 快速实验迭代，减少工程负担

---

#### 方案 C：混合使用（推荐）

**SFT 阶段**: 使用 ms-swift CLI 快速跑 baseline
```bash
# 快速验证
swift sft \
    --model Qwen/Qwen3-4B \
    --tuner_type lora \
    --dataset 'kdeng03/mol-rep-conversion-v0' \
    --lora_rank 16 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --num_train_epochs 2 \
    --learning_rate 1e-4 \
    --output_dir ckpt/mol-sft-baseline
```

**GRPO 阶段**: 使用 ms-swift 内置 RL
```bash
swift rlhf \
    --rlhf_type grpo \
    --model Qwen/Qwen3-4B \
    --tuner_type lora \
    --dataset 'kdeng03/mol-rep-conversion-v0' \
    --use_vllm true \
    --vllm_mode colocate \
    --output_dir ckpt/mol-grpo
```

**自定义实验**: 保留 TRL 脚本做高度定制化的 ablation
```bash
# 用现有的 sft.py 做精细控制
python scripts/sft.py \
    --config configs/model_training_schema.yaml \
    --custom_task_mix ...
```

---

### 数据格式转换建议

如果选择 ms-swift，需要将当前数据转换为标准格式：

```python
# 当前数据格式（推测）
{"prompt": "...", "completion": "...", "task_type": "...", "mol_id": "..."}

# 转换为 ms-swift 标准格式
{"messages": [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": completion}
], "mol_id": "...", "task_type": "..."}
```

**转换脚本示例**:
```python
from datasets import load_dataset

ds = load_dataset("kdeng03/mol-rep-conversion-v0", split="train")

def to_messages(example):
    return {
        "messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["completion"]}
        ],
        "mol_id": example.get("mol_id", ""),
        "task_type": example.get("task_type", ""),
    }

ds_messages = ds.map(to_messages)
ds_messages.to_json("mol-rep-conversion-messages.jsonl", force_ascii=False)
```

---

### 总结

| 阶段 | 推荐框架 | 理由 |
|------|---------|------|
| **Baseline 实验** | ms-swift | 快速启动，减少工程负担 |
| **Ablation 实验** | TRL（现有代码） | 需要精细控制数据混合 |
| **GRPO/RL** | ms-swift | 内置算法，开箱即用 |
| **推理评估** | ms-swift | 内置 vLLM 加速 + evalscope |

**最终建议**: 先用 ms-swift 跑通 baseline，再用 TRL 做精细化 ablation 实验。两者可以共存，互不冲突。

## 6. 数据下载问题：ms-swift 必须下载到本地吗？

### 核心问题：OCR 数据含图片，不想下载到本地，不方便迁移

**答案：ms-swift 完全支持内存中加载 HF dataset，和 TRL 一样灵活。**

---

### 方案一：Python API + `load_dataset`（和 TRL 完全一样）

ms-swift 的 Python API 底层也是用 `datasets.load_dataset()`，所以 **TRL 能做的，ms-swift 都能做**：

```python
from datasets import load_dataset
from swift import sft_main, SftArguments

# 1. 在内存中加载 HF dataset（图片按需流式读取，不存本地）
ds = load_dataset("kdeng03/mol-rep-ocr-v0", split="train")

# 2. 自定义数据处理逻辑（全部在内存中）
def to_messages(example):
    return {
        "messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["completion"]}
        ],
        "images": [example["mol_image"]],  # PIL Image 对象，直接从 HF 流式读取
    }

ds = ds.map(to_messages)

# 3. 用 ms-swift 训练（传入内存中的 dataset）
# 注意：sft_main 目前主要支持 CLI 参数传入 dataset ID/path
# 但可以通过 register_dataset 注册内存 dataset
```

---

### 方案二：`register_dataset` + `load_function`（ms-swift 特有）

ms-swift 提供了 `register_dataset` 机制，可以自定义数据加载函数：

```python
from swift.dataset import DatasetMeta, register_dataset, load_dataset
from datasets import load_dataset as hf_load_dataset

# 自定义加载函数：从 HF 流式读取，不存本地
def load_mol_ocr_dataset():
    # 从 HF 流式加载，图片按需读取
    ds = hf_load_dataset("kdeng03/mol-rep-ocr-v0", split="train", streaming=True)
    
    def to_messages(example):
        return {
            "messages": [
                {"role": "user", "content": example["prompt"]},
                {"role": "assistant", "content": example["completion"]}
            ],
            "images": [example["mol_image"]],
        }
    
    return ds.map(to_messages)

# 注册数据集
register_dataset(DatasetMeta(
    dataset_name="mol-ocr-stream",
    load_function=load_mol_ocr_dataset,
))

# 使用
train_ds = load_dataset("mol-ocr-stream")[0]
```

---

### 方案三：CLI + `--streaming true`（最简单）

```bash
# 流式读取 HF dataset，不缓存到本地
swift sft \
    --model Qwen/Qwen3-4B-Instruct \
    --tuner_type lora \
    --dataset kdeng03/mol-rep-ocr-v0 \
    --use_hf true \
    --streaming true
```

**行为**:
- 边训练边从 HF 拉取数据
- **图片不缓存到本地**，按需流式读取
- 不占用本地磁盘空间
- 适合你的场景：OCR 数据含图片，不想下载到本地

---

### 方案四：`--lazy_tokenize true`（多模态推荐）

```bash
swift sft \
    --dataset kdeng03/mol-rep-ocr-v0 \
    --use_hf true \
    --streaming true \
    --lazy_tokenize true
```

**行为**:
- 训练时才 tokenize，预处理时不加载图片
- 避免训练前加载所有图片到内存
- **OCR 任务强烈推荐**

---

### TRL vs ms-swift 数据加载对比

| 能力 | TRL | ms-swift |
|------|-----|----------|
| `load_dataset()` 内存加载 | ✅ | ✅（底层一样） |
| Streaming 模式 | ✅ | ✅ `--streaming true` |
| 自定义数据处理 | ✅ 自己写 `map` | ✅ `register_dataset` + `load_function` |
| 图片流式读取 | ✅ | ✅ |
| 不缓存到本地 | ✅ | ✅ |

**结论**: ms-swift 在数据加载灵活性和 TRL 是一样的，底层都用 `datasets` 库。你完全可以在内存中处理所有数据逻辑，不下载到本地。

---

### 针对你的场景的最佳实践

```python
# train_mol_ocr.py — 所有数据处理逻辑在代码中，不依赖本地文件
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from datasets import load_dataset
from swift import sft_main, SftArguments
from swift.dataset import DatasetMeta, register_dataset, load_dataset as swift_load_dataset

# 1. 注册自定义数据集（内存中处理）
def load_mol_conversion():
    """纯文本 conversion 数据"""
    ds = load_dataset("kdeng03/mol-rep-conversion-v0", split="train")
    
    def to_messages(example):
        return {
            "messages": [
                {"role": "user", "content": example["prompt"]},
                {"role": "assistant", "content": example["completion"]}
            ]
        }
    
    return ds.map(to_messages)

def load_mol_ocr():
    """OCR 数据（含图片），流式读取"""
    ds = load_dataset("kdeng03/mol-rep-ocr-v0", split="train", streaming=True)
    
    def to_messages(example):
        return {
            "messages": [
                {"role": "user", "content": example["prompt"]},
                {"role": "assistant", "content": example["completion"]}
            ],
            "images": [example["mol_image"]],
        }
    
    return ds.map(to_messages)

# 注册
register_dataset(DatasetMeta(
    dataset_name="mol-conversion",
    load_function=load_mol_conversion,
))
register_dataset(DatasetMeta(
    dataset_name="mol-ocr",
    load_function=load_mol_ocr,
))

# 2. 训练（使用注册的数据集名称）
result = sft_main(SftArguments(
    model='Qwen/Qwen3-4B-Instruct',
    tuner_type='lora',
    dataset=['mol-conversion'],  # 使用注册名称
    lora_rank=16,
    lora_alpha=32,
    target_modules='all-linear',
    num_train_epochs=2,
    learning_rate=1e-4,
    output_dir='ckpt/mol-sft',
))
```

**优势**:
- 所有数据处理逻辑在代码中，不依赖本地文件
- 图片流式读取，不下载到本地
- 方便迁移：代码 + 配置即可，不需要带数据文件
- 和 TRL 的灵活性完全一致